# Modelos de Bagging

Se comparan Bagging CART, Random Forest y Extra Trees. El número de árboles se selecciona con validación sobre entrenamiento y el conjunto de prueba se reserva para la evaluación final.


## Librerías


In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import BaggingClassifier, BaggingRegressor, ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor


## Datos procesados


In [ ]:
bank_train_con_duration = pd.read_csv("data/processed/serie_computacional/bank_train_con_duration.csv")
bank_test_con_duration = pd.read_csv("data/processed/serie_computacional/bank_test_con_duration.csv")
bank_train_sin_duration = pd.read_csv("data/processed/serie_computacional/bank_train_sin_duration.csv")
bank_test_sin_duration = pd.read_csv("data/processed/serie_computacional/bank_test_sin_duration.csv")

consumo_train = pd.read_csv("data/processed/serie_computacional/power_train_hourly.csv")
consumo_test = pd.read_csv("data/processed/serie_computacional/power_test_hourly.csv")

resumen_datos = pd.DataFrame([
    {"Dataset": "Bank Marketing", "Escenario": "con duration", "Entrenamiento": len(bank_train_con_duration), "Prueba": len(bank_test_con_duration), "Variables": bank_train_con_duration.shape[1] - 1},
    {"Dataset": "Bank Marketing", "Escenario": "sin duration", "Entrenamiento": len(bank_train_sin_duration), "Prueba": len(bank_test_sin_duration), "Variables": bank_train_sin_duration.shape[1] - 1},
    {"Dataset": "Electric Power", "Escenario": "regresión horaria", "Entrenamiento": len(consumo_train), "Prueba": len(consumo_test), "Variables": consumo_train.shape[1] - 2},
])

resumen_datos


## Funciones auxiliares


In [ ]:
def preparar_preprocesador(X):
    variables_numericas = [col for col in X.columns if pd.api.types.is_numeric_dtype(X[col])]
    variables_categoricas = [col for col in X.columns if col not in variables_numericas]

    try:
        codificador = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        codificador = OneHotEncoder(handle_unknown="ignore", sparse=False)

    pasos = []
    if variables_numericas:
        pasos.append(("numericas", SimpleImputer(strategy="median"), variables_numericas))
    if variables_categoricas:
        pasos.append(("categoricas", codificador, variables_categoricas))

    return ColumnTransformer(pasos), variables_numericas, variables_categoricas


def separar_bank(entrenamiento, prueba):
    X_entrenamiento = entrenamiento.drop(columns=["y"])
    y_entrenamiento = entrenamiento["y"]
    X_prueba = prueba.drop(columns=["y"])
    y_prueba = prueba["y"]
    return X_entrenamiento, y_entrenamiento, X_prueba, y_prueba


def separar_consumo(entrenamiento, prueba):
    objetivo = "Global_active_power"
    columnas_excluidas = [objetivo, "datetime"]
    X_entrenamiento = entrenamiento.drop(columns=columnas_excluidas)
    y_entrenamiento = entrenamiento[objetivo]
    X_prueba = prueba.drop(columns=columnas_excluidas)
    y_prueba = prueba[objetivo]
    return X_entrenamiento, y_entrenamiento, X_prueba, y_prueba


def evaluar_clasificacion(modelo, nombre, X_entrenamiento, y_entrenamiento, X_prueba, y_prueba, escenario):
    preprocesador, numericas, categoricas = preparar_preprocesador(X_entrenamiento)
    flujo = Pipeline([("preprocesamiento", preprocesador), ("modelo", modelo)])
    inicio = time.perf_counter()
    flujo.fit(X_entrenamiento, y_entrenamiento)
    tiempo_ajuste = time.perf_counter() - inicio
    predicciones = flujo.predict(X_prueba)
    probabilidades = flujo.predict_proba(X_prueba)[:, 1] if hasattr(flujo, "predict_proba") else None

    return flujo, {
        "Dataset": "Bank Marketing",
        "Escenario": escenario,
        "Modelo": nombre,
        "F1 de la clase positiva": f1_score(y_prueba, predicciones, pos_label=1, zero_division=0),
        "Precisión": precision_score(y_prueba, predicciones, pos_label=1, zero_division=0),
        "Sensibilidad": recall_score(y_prueba, predicciones, pos_label=1, zero_division=0),
        "AUC-ROC": roc_auc_score(y_prueba, probabilidades) if probabilidades is not None else np.nan,
        "Exactitud": accuracy_score(y_prueba, predicciones),
        "Tiempo de ajuste (s)": tiempo_ajuste,
        "Variables originales": X_entrenamiento.shape[1],
        "Variables numéricas": len(numericas),
        "Variables categóricas": len(categoricas),
    }


def evaluar_regresion(modelo, nombre, X_entrenamiento, y_entrenamiento, X_prueba, y_prueba):
    preprocesador, numericas, categoricas = preparar_preprocesador(X_entrenamiento)
    flujo = Pipeline([("preprocesamiento", preprocesador), ("modelo", modelo)])
    inicio = time.perf_counter()
    flujo.fit(X_entrenamiento, y_entrenamiento)
    tiempo_ajuste = time.perf_counter() - inicio
    predicciones = flujo.predict(X_prueba)
    mse = mean_squared_error(y_prueba, predicciones)

    return flujo, {
        "Dataset": "Individual Household Electric Power Consumption",
        "Modelo": nombre,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_prueba, predicciones),
        "MSE": mse,
        "R²": r2_score(y_prueba, predicciones),
        "Tiempo de ajuste (s)": tiempo_ajuste,
        "Variables originales": X_entrenamiento.shape[1],
        "Variables numéricas": len(numericas),
        "Variables categóricas": len(categoricas),
    }


## Selección del número de árboles

En clasificación se usa validación cruzada estratificada con tres particiones y F1 de la clase positiva. En regresión se usa `TimeSeriesSplit` para respetar el orden temporal y RMSE como criterio.


In [ ]:
numero_arboles_probados = [25, 50, 100, 200]
validacion_clasificacion = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
validacion_regresion = TimeSeriesSplit(n_splits=3)


def modelos_bagging_clasificacion(numero_arboles):
    return {
        "Bagging CART": BaggingClassifier(
            estimator=DecisionTreeClassifier(class_weight="balanced", random_state=42),
            n_estimators=numero_arboles,
            random_state=42,
            n_jobs=1,
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=numero_arboles,
            class_weight="balanced",
            random_state=42,
            n_jobs=1,
            bootstrap=True,
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=numero_arboles,
            class_weight="balanced",
            random_state=42,
            n_jobs=1,
        ),
    }


def modelos_bagging_regresion(numero_arboles):
    return {
        "Bagging CART": BaggingRegressor(
            estimator=DecisionTreeRegressor(random_state=42),
            n_estimators=numero_arboles,
            random_state=42,
            n_jobs=1,
        ),
        "Random Forest": RandomForestRegressor(
            n_estimators=numero_arboles,
            random_state=42,
            n_jobs=1,
            bootstrap=True,
        ),
        "Extra Trees": ExtraTreesRegressor(
            n_estimators=numero_arboles,
            random_state=42,
            n_jobs=1,
        ),
    }


def puntuacion_media(modelo, X, y, validacion, metrica):
    preprocesador, _, _ = preparar_preprocesador(X)
    flujo = Pipeline([("preprocesamiento", preprocesador), ("modelo", modelo)])
    puntuaciones = cross_val_score(flujo, X, y, cv=validacion, scoring=metrica, n_jobs=None)
    return float(puntuaciones.mean())


## Clasificación


In [ ]:
filas_validacion = []
filas_test = []

for escenario, entrenamiento, prueba in [
    ("con duration", bank_train_con_duration, bank_test_con_duration),
    ("sin duration", bank_train_sin_duration, bank_test_sin_duration),
]:
    X_entrenamiento, y_entrenamiento, X_prueba, y_prueba = separar_bank(entrenamiento, prueba)
    for nombre_modelo in ["Bagging CART", "Random Forest", "Extra Trees"]:
        inicio = time.perf_counter()
        resultados_modelo = []
        for numero_arboles in numero_arboles_probados:
            modelo = modelos_bagging_clasificacion(numero_arboles)[nombre_modelo]
            media_validacion = puntuacion_media(
                modelo, X_entrenamiento, y_entrenamiento, validacion_clasificacion, "f1"
            )
            resultados_modelo.append((numero_arboles, media_validacion))
        mejor_numero, mejor_media = max(resultados_modelo, key=lambda item: item[1])
        tiempo_validacion = time.perf_counter() - inicio
        filas_validacion.append({
            "Tarea": "clasificación",
            "Escenario": escenario,
            "Modelo": nombre_modelo,
            "Números de árboles probados": str(numero_arboles_probados),
            "Mejor número de árboles": mejor_numero,
            "Métrica de validación": "F1 clase positiva",
            "Media de validación": mejor_media,
            "Tiempo de validación (s)": tiempo_validacion,
        })
        modelo_final = modelos_bagging_clasificacion(mejor_numero)[nombre_modelo]
        _, fila_test = evaluar_clasificacion(
            modelo_final, nombre_modelo, X_entrenamiento, y_entrenamiento, X_prueba, y_prueba, escenario
        )
        fila_test["Número de árboles"] = mejor_numero
        filas_test.append(fila_test)

seleccion_bagging_clasificacion = pd.DataFrame(filas_validacion)
resultados_bagging_clasificacion = pd.DataFrame(filas_test)

display(seleccion_bagging_clasificacion)
resultados_bagging_clasificacion


## Regresión


In [ ]:
filas_validacion = []
filas_test = []
X_entrenamiento, y_entrenamiento, X_prueba, y_prueba = separar_consumo(consumo_train, consumo_test)

for nombre_modelo in ["Bagging CART", "Random Forest", "Extra Trees"]:
    inicio = time.perf_counter()
    resultados_modelo = []
    for numero_arboles in numero_arboles_probados:
        modelo = modelos_bagging_regresion(numero_arboles)[nombre_modelo]
        media_validacion = -puntuacion_media(
            modelo, X_entrenamiento, y_entrenamiento, validacion_regresion, "neg_root_mean_squared_error"
        )
        resultados_modelo.append((numero_arboles, media_validacion))
    mejor_numero, mejor_media = min(resultados_modelo, key=lambda item: item[1])
    tiempo_validacion = time.perf_counter() - inicio
    filas_validacion.append({
        "Tarea": "regresión",
        "Escenario": "consumo eléctrico horario",
        "Modelo": nombre_modelo,
        "Números de árboles probados": str(numero_arboles_probados),
        "Mejor número de árboles": mejor_numero,
        "Métrica de validación": "RMSE",
        "Media de validación": mejor_media,
        "Tiempo de validación (s)": tiempo_validacion,
    })
    modelo_final = modelos_bagging_regresion(mejor_numero)[nombre_modelo]
    _, fila_test = evaluar_regresion(modelo_final, nombre_modelo, X_entrenamiento, y_entrenamiento, X_prueba, y_prueba)
    fila_test["Número de árboles"] = mejor_numero
    filas_test.append(fila_test)

seleccion_bagging_regresion = pd.DataFrame(filas_validacion)
resultados_bagging_regresion = pd.DataFrame(filas_test)

display(seleccion_bagging_regresion)
resultados_bagging_regresion


## Guardado y figuras


In [ ]:
Path("reports/tables/serie_computacional").mkdir(parents=True, exist_ok=True)
Path("reports/figures/serie_computacional").mkdir(parents=True, exist_ok=True)

seleccion_bagging_clasificacion.to_csv("reports/tables/serie_computacional/bagging_validacion_clasificacion.csv", index=False)
seleccion_bagging_regresion.to_csv("reports/tables/serie_computacional/bagging_validacion_regresion.csv", index=False)
resultados_bagging_clasificacion.to_csv("reports/tables/serie_computacional/bagging_clasificacion_metricas.csv", index=False)
resultados_bagging_regresion.to_csv("reports/tables/serie_computacional/bagging_regresion_metricas.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
grafico = resultados_bagging_clasificacion.copy()
grafico["Etiqueta"] = grafico["Modelo"] + "\n" + grafico["Escenario"]
ax.bar(grafico["Etiqueta"], grafico["F1 de la clase positiva"], color="#4c78a8")
ax.set_title("Modelos de Bagging: F1 de la clase positiva")
ax.set_ylabel("F1 de la clase positiva")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig("reports/figures/serie_computacional/bagging_clasificacion_f1.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(resultados_bagging_regresion["Modelo"], resultados_bagging_regresion["RMSE"], color="#f58518")
ax.set_title("Modelos de Bagging: RMSE")
ax.set_ylabel("RMSE")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig("reports/figures/serie_computacional/bagging_regresion_rmse.png", dpi=150)
plt.show()

pd.DataFrame([
    {"archivo": "bagging_validacion_clasificacion.csv"},
    {"archivo": "bagging_validacion_regresion.csv"},
    {"archivo": "bagging_clasificacion_metricas.csv"},
    {"archivo": "bagging_regresion_metricas.csv"},
])
